In [ ]:
# ========== 导入：第 6 周定价者（Pricer）顶点练习要用的库 ==========

# os：拼路径、读环境变量、写绝对路径
import os
# re：正则（本格先导入，后续格可能用到）
import re
# json：把 messages 字典序列化进 JSONL
import json
# random：平衡采样时设种子、sample、shuffle
import random
# time：计时相关（本练习导入保留）
import time
# sys：把 week6 目录加入模块搜索路径
import sys
# defaultdict：按价格桶收集 Item 列表
from collections import defaultdict
# load_dotenv：从 .env 读密钥
from dotenv import load_dotenv
# Hugging Face Hub 登录（HF_TOKEN）
from huggingface_hub import login
# OpenAI 客户端：后续微调/评测流程可能用到
from openai import OpenAI

# ===================================================================
# 定价者：顶点练习（第 6 周）
# THE PRICER: CAPSTONE EXERCISE (WEEK 6)
# ===================================================================
# 该脚本通过以下方式改进了讲师的第 5 天微调模型：
# This script improves upon the instructor's Day 5 fine-tuned model by:
# 1. 平衡抽样：确保法学硕士了解 500 美元的电视，而不仅仅是 5 美元的电缆。
# 1. Balanced Sampling: Ensuring the LLM learns about $500 TVs, not just $5 cables.
# 2. 专家角色：为人工智能设计特定领域的提示。
# 2. Expert Persona: Engineering a domain-specific prompt for the AI.


## 第 0 步：设置和导入

先准备环境：导入库、加载 `.env`、把课程 `week6/pricer` 放进 `sys.path`，再初始化 OpenAI / Hugging Face。


## 从仓库根目录加载 `.env`

`load_dotenv(override=True)` 会把项目根附近的 `.env` 读进环境变量（含 API Key、`HF_TOKEN`）。


In [ ]:
# ========== 加载环境变量 + 导入 pricer + 初始化客户端 ==========

# 读 .env；override=True 覆盖已有同名环境变量
load_dotenv(override=True)

# 把相对路径 ../../../week6 加成绝对路径后 append 到 sys.path，才能 import 讲师自定义类
sys.path.append(os.path.abspath("../../../week6"))
try:
    # Item：商品数据模型；evaluate / Tester：课程评测工具
    from pricer.items import Item
    from pricer.evaluator import evaluate, Tester
except ImportError:
    # 导入失败时给出提示（英文 warning 字符串保持原样）
    print("Warning: Could not import pricer modules. Make sure you are running this from the community-contributions folder.")

# 初始化 OpenAI 客户端（默认读 OPENAI_API_KEY）
openai = OpenAI()
# 从环境变量取 Hugging Face token
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    # 登录 HF Hub，便于后面 from_hub 拉数据集；顺便写 git credential
    login(hf_token, add_to_git_credential=True)
else:
    # 没找到 token 时警告（文案保持原样）
    print("WARNING: HF_TOKEN not found in .env")


## 第 1 步：加载精简数据集

从 Hugging Face Hub 拉取课程清理好的轻量商品集（`items_lite`），得到 train / val / test。


## 加载数据集提示

下一格会打印类似 `Loaded … training items …`；若失败，先检查 `HF_TOKEN` 与网络。


In [ ]:
# ========== 从 Hugging Face Hub 拉取 items_lite ==========

# 数据集作者命名空间（保持原字符串）
username = "ed-donner"
# 拼出完整数据集名：ed-donner/items_lite
dataset_name = f"{username}/items_lite"

# test≈1000、val≈1000、train≈20000：规模量级提示（逻辑仍是 from_hub 一次拉齐）
train, val, test = Item.from_hub(dataset_name)
# 打印三类集合的条数（千分位）
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


## 第 2 步：第一个改进 — 平衡采样（Balanced Sampling）

若只取训练集「前 N 条」，模型可能只见过廉价配件、没见过贵价电器。  
做法：按价格分桶（bucket），每桶抽同样多的样本，再打乱。


In [ ]:
# ========== 改进 1：按价格分桶做平衡采样 ==========
# 缺陷：如果前 100 件都是廉价手机壳，模型永远学不到贵价商品长什么样
# 解决：把价格分成若干桶，再从每个桶里大致均等地抽取

# 把连续价格映射到离散桶标签（字符串必须原样返回）
def categorize_price(price):
    # 低价段：50 美元以下
    if price < 50:
        return '$0-50'
    # 中低价段
    elif price < 150:
        return '$50-150'
    # 中高价段
    elif price < 300:
        return '$150-300'
    # 高价段：300 及以上
    else:
        return '$300+'

# 进度提示（英文 STEP 文案保持原样）
print("\n--- STEP 2: Creating Balanced Training Data ---")
# 用 defaultdict(list) 按桶收集训练 Item
price_buckets = defaultdict(list)
# 遍历全部训练集，逐条归桶
for item in train:
    # 根据价格归桶
    bucket = categorize_price(item.price)
    # 把该 Item 追加到对应桶的列表
    price_buckets[bucket].append(item)

# 目标约 100 条微调样本：4 个桶 × 每桶 25 条
ITEMS_PER_BUCKET = 25
# 累积最终微调训练集
fine_tune_train = []
# 固定随机种子，保证可复现
random.seed(42)

# 逐桶抽样
for bucket, items_in_bucket in price_buckets.items():
    # 桶内不够 25 就有多少取多少
    sample_size = min(ITEMS_PER_BUCKET, len(items_in_bucket))
    # 无放回随机抽样
    sample = random.sample(items_in_bucket, sample_size)
    # 并入总训练列表
    fine_tune_train.extend(sample)

# 打乱，避免训练时按价格区间成块出现
random.shuffle(fine_tune_train)

# 验证集也做类似的分桶采样，目标约 50 条
val_buckets = defaultdict(list)
# 遍历验证集归桶
for item in val:
    bucket = categorize_price(item.price)
    val_buckets[bucket].append(item)

# 累积微调验证集
fine_tune_validation = []
for bucket, items_in_bucket in val_buckets.items():
    # 每桶最多取 13 条
    sample = random.sample(items_in_bucket, min(13, len(items_in_bucket)))
    fine_tune_validation.extend(sample)
# 裁到正好 50 条
fine_tune_validation = fine_tune_validation[:50] # Trim exactly to 50

# 打印训练集规模
print(f"Created a balanced training set of {len(fine_tune_train)} items.")
# 按桶名排序后统计每桶实际抽到多少条，确认是否平衡
for bucket in sorted(price_buckets.keys()):
    count = sum(1 for x in fine_tune_train if categorize_price(x.price) == bucket)
    print(f" - {bucket} bucket: {count} items")


## 第 3 步：第二个改进 — 专家角色（Expert Persona）

把 system prompt 从「泛泛估价」升级成「资深定价分析师」角色，并严格约束回复格式 `Price is $XX.XX`。


In [ ]:
# ========== 改进 2：专家角色 system prompt + 构造微调 messages ==========

# 专家人设 + 只允许特定价格格式（整段英文 prompt 必须原样保留）
EXPERT_SYSTEM_PROMPT = """You are a senior pricing analyst with deep expertise in consumer electronics, appliances, and retail goods.
Analyze the brand, product specifications, and features to estimate the most likely retail market price in USD.
Respond ONLY with the format: Price is $XX.XX"""

# 把单个 Item 变成微调用的「闪卡」messages：正面描述、背面真价
def messages_for(item):
    """Creates the 'Flashcard' for the LLM. Front = Description, Back = Price"""
    # 清理讲师 test_prompt 里多余短语，得到更干净的 user 面
    user_prompt = item.test_prompt().replace(" to the nearest dollar", "").replace("\n\nPrice is $", "")
    # 返回 OpenAI messages 三元组：system / user / assistant
    return [
        # system：专家角色与输出格式约束
        {"role": "system", "content": EXPERT_SYSTEM_PROMPT},
        # user：商品描述（卡片正面）
        {"role": "user", "content": user_prompt},
        # assistant：真价答案（卡片背面；微调用）
        {"role": "assistant", "content": f"Price is ${item.price:.2f}"} # The answer on the back
    ]


## 第 4 步：格式化并保存 JSONL 文件

把每条 `messages` 写成 OpenAI 微调常用的 JSONL：一行一个 `{"messages": [...]}` 对象。


In [ ]:
# ========== 写成本地 JSONL：训练集 + 验证集 ==========

def make_jsonl(items):
    # 累积多行 JSON 文本
    result = ""
    for item in items:
        # messages_for → json.dumps；外层再包 {"messages": ...} 并换行
        result += '{"messages": ' + json.dumps(messages_for(item)) + '}\n'
    # 去掉末尾多余换行
    return result.strip()

def write_local(items, filename):
    # 以写模式打开目标文件，写入整份 JSONL
    with open(filename, "w") as f:
        f.write(make_jsonl(items))
    # 返回绝对路径，方便后面上传/核对
    return os.path.abspath(filename)

# 写出平衡后的训练 JSONL
train_path = write_local(fine_tune_train, "fine_tune_train.jsonl")
# 写出平衡后的验证 JSONL
val_path = write_local(fine_tune_validation, "fine_tune_validation.jsonl")

# 成功提示（含 emoji 的原文字符串保持不变）
print(f"✅ Successfully wrote structured training data to:\n -> {train_path}")
print(f"✅ Successfully wrote structured validation data to:\n -> {val_path}")
